In [ ]:
import requests
class SEADParaibaAPI:
    def __init__(self):
        self.base_url = "https://api.dadosabertos.codata.pb.gov.br/api/v1/remuneracao"
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/json",
            "Content-Type": "application/json"
        })

    def listar_servidores(self, ano, mes, pagina=1):
        url = f"{self.base_url}/servidor"
        
        params = {
            "ano": str(ano),
            "mes": str(mes).zfill(2), 
            "page": pagina
        }
        
        print(f"Buscando URL EXATA: {url}")
        print(f"Parâmetros enviados: {params}...")
        
        response = self.session.get(url, params=params)
        
        if response.status_code != 200:
            print(f"Erro na API: {response.status_code}")
            print(response.text)
            return None

        return response.json()
    
    def consultar_remuneracao(self, ano, mes, nome_servidor=None):
        """Consulta os dados da folha de pagamento do servidor."""
        url = f"{self.base_url}/servidor/{ano}/{mes}"
        params = {}
        
        if nome_servidor:
            params["nome"] = nome_servidor
            
        response = self.session.get(url, params=params)
        response.raise_for_status()
        return response.json()

    def consultar_diarias(self, ano, nome_servidor=None):
        """Consulta as diárias emitidas para o servidor no ano vigente."""
        url = f"{self.base_url}/diarias"
        params = {"anoExercicio": ano}
        
        if nome_servidor:
            params["favorecido"] = nome_servidor
            
        response = self.session.get(url, params=params)
        response.raise_for_status()
        return response.json()

    def extrair_dossiê_servidor(self, ano, mes, nome_alvo):
        """Busca e consolida os dados de remuneração e diárias de um servidor."""
        print(f"Buscando dados de: {nome_alvo} ({mes}/{ano})...")
        
        dados_remuneracao = self.consultar_remuneracao(ano, mes, nome_alvo)
        
        dados_diarias = self.consultar_diarias(ano, nome_alvo)

        if not dados_remuneracao:
            print("Nenhum dado de remuneração encontrado.")
            return

        for registro in dados_remuneracao:
            if nome_alvo.upper() in str(registro.get("nome", "")).upper():
                
                total_diarias = sum(
                    float(d.get("valor", 0)) 
                    for d in dados_diarias 
                    if nome_alvo.upper() in str(d.get("nomeFavorecido", "")).upper()
                )
                
                dossie = {
                    "Órgão de lotação": registro.get("orgaoLotacao", "N/A"),
                    "Cargo": registro.get("cargo", "N/A"),
                    "Salário bruto": float(registro.get("remuneracaoBruta", 0)),
                    "Salário líquido": float(registro.get("remuneracaoLiquida", 0)),
                    "Gratificações": float(registro.get("gratificacoes", 0)),
                    "Vantagens (Pessoais/Indenizatórias)": float(registro.get("vantagens", 0)),
                    "Diárias recebidas (Acumulado do Ano)": total_diarias
                }
                
                self._imprimir_relatorio(nome_alvo, dossie)
                return

    def _imprimir_relatorio(self, nome, dossie):
        print(f"\n--- RELATÓRIO: {nome.upper()} ---")
        for chave, valor in dossie.items():
            if isinstance(valor, float):
                print(f"{chave}: R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
            else:
                print(f"{chave}: {valor}")
        print("-----------------------------------")


if __name__ == "__main__":
    api = SEADParaibaAPI()
    
    dados = api.listar_servidores(ano=2025, mes=10, pagina=1)
    
    if dados:
        print("\n--- ESTRUTURA DO RETORNO DA API ---")
        print("Tipo do objeto:", type(dados))
        
        if isinstance(dados, dict):
            print("Chaves principais do JSON:", dados.keys())
            
            for chave, valor in dados.items():
                if isinstance(valor, list):
                    print(f"\nAchei a lista! Ela está dentro da chave: '{chave}'")
                    print(f"Quantidade de registros nesta página: {len(valor)}")
                    print("\nPrimeiro registro da lista para vermos os campos:")
                    print(valor[0])
                    break
            else:
                print("\nNenhuma lista encontrada diretamente nas chaves principais. Amostra do conteúdo:")
                print(str(dados)[:500])
                
        elif isinstance(dados, list):
            print("É uma lista direta! Primeiro item:")
            print(dados[0])

In [ ]:
import requests
import csv
import os

class SEADParaibaCSV:
    def __init__(self):
        self.base_url = "https://api.dadosabertos.codata.pb.gov.br/api/v1/remuneracao"
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/json",
            "Content-Type": "application/json"
        })

    def buscar_pagina(self, ano, mes, pagina):
        """Busca uma página específica da API."""
        url = f"{self.base_url}/servidor"
        params = {
            "ano": str(ano),
            "mes": str(mes).zfill(2),
            "page": pagina
        }
        
        response = self.session.get(url, params=params)
        
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Erro ao buscar página {pagina}. Status: {response.status_code}")
            return None

    def localizar_lista(self, dados_json):
        """Descobre dinamicamente em qual chave a API guardou a lista de servidores."""
        if isinstance(dados_json, list):
            return dados_json
            
        if isinstance(dados_json, dict):
            for chave, valor in dados_json.items():
                if isinstance(valor, list):
                    return valor
        return []
    def exportar_para_csv(self, ano, mes, max_paginas=5):
            """
            Busca os dados e salva em um arquivo CSV contendo todas as colunas retornadas pela API.
            `max_paginas` define quantas páginas da folha baixar.
            """
            nome_arquivo = f"servidores_pb_{ano}_{str(mes).zfill(2)}.csv"
            
            print(f"Iniciando extração. Os dados serão salvos em: {nome_arquivo}\n")

            # Variáveis de controle para inicialização tardia do arquivo CSV (para descobrir os cabeçalhos dinamicamente)
            arquivo_csv = None
            escritor = None
            cabecalhos = []

            for pagina in range(1, max_paginas + 1):
                print(f"Baixando página {pagina}...")
                dados = self.buscar_pagina(ano, mes, pagina)
                
                if not dados:
                    break
                    
                lista_servidores = self.localizar_lista(dados)
                
                if not lista_servidores:
                    print("A lista de servidores veio vazia ou acabaram as páginas.")
                    break
                    
                import json
                # Processa cada servidor da página atual
                for servidor in lista_servidores:
                    print(f"Processando servidor: {json.dumps(servidor)}...")
                    
                    # Se for o primeiro registro encontrado, extraímos todas as chaves para montar os cabeçalhos dinamicamente
                    if not cabecalhos:
                        cabecalhos = list(servidor.keys())
                        
                        # Abre o arquivo CSV para escrita usando os cabeçalhos dinâmicos
                        arquivo_csv = open(nome_arquivo, mode="w", newline="", encoding="utf-8-sig")
                        escritor = csv.DictWriter(arquivo_csv, fieldnames=cabecalhos, delimiter=";")
                        escritor.writeheader()

                    # Escreve a linha mapeando todas as chaves diretamente do dicionário da API
                    escritor.writerow(servidor)

            # Garante o fechamento correto do arquivo caso ele tenha sido aberto
            if arquivo_csv:
                arquivo_csv.close()

            print(f"\nExtração concluída com sucesso! Arquivo salvo no caminho: {os.path.abspath(nome_arquivo)}")
# --- Execução do Código ---
if __name__ == "__main__":
    extrator = SEADParaibaCSV()
    
    # Extraindo dados de Outubro de 2025
    # Por padrão, configurei max_paginas=2 para teste rápido. 
    # Para baixar o estado inteiro, mude para um valor alto, ex: max_paginas=5000
    extrator.exportar_para_csv(ano=2026, mes=10, max_paginas=2)

In [ ]:
import pandas as pd
import plotly.express as px
import nbformat

# Carregar CSV
df = pd.read_csv(
    "folha.csv",
    sep=";",
    decimal=".",
    encoding="utf-8"
)

# Converter valores numéricos
colunas_numericas = [
    "vantagemFixa",
    "vantagemVariavel",
    "valorBruto",
    "valorPrevidenciario",
    "valorIr",
    "descontoObrigatorio",
    "valorDesconto",
    "valorLiquido"
]

for col in colunas_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")


fig = px.histogram(
    df,
    x="valorLiquido",
    nbins=50,
    title="Distribuição dos Salários Líquidos"
)
fig.show()


top_cargos = (
    df.groupby("nomeCargo")["valorLiquido"]
    .mean()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

fig = px.bar(
    top_cargos,
    x="valorLiquido",
    y="nomeCargo",
    orientation="h",
    title="Top 20 Cargos por Média Salarial"
)

fig.show()


top_unidades = (
    df.groupby("nomeUnidadeTrabalho")["valorLiquido"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

fig = px.bar(
    top_unidades,
    x="valorLiquido",
    y="nomeUnidadeTrabalho",
    orientation="h",
    title="Top 20 Unidades por Gasto"
)

fig.show()


regime = (
    df.groupby("regimeContratual")
    .size()
    .reset_index(name="quantidade")
)

fig = px.pie(
    regime,
    names="regimeContratual",
    values="quantidade",
    title="Regime Contratual"
)

fig.show()


fig = px.box(
    df,
    x="sexo",
    y="valorLiquido",
    title="Distribuição Salarial por Sexo"
)

fig.show()


gastos_orgao = (
    df.groupby("orgaoLotacao")["valorLiquido"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

fig = px.bar(
    gastos_orgao,
    x="valorLiquido",
    y="orgaoLotacao",
    orientation="h",
    title="Gasto Total por Órgão"
)

fig.show()


evolucao = (
    df.groupby("periodo")["valorLiquido"]
    .sum()
    .reset_index()
)

fig = px.line(
    evolucao,
    x="periodo",
    y="valorLiquido",
    markers=True,
    title="Evolução da Folha"
)

fig.show()


fig = px.box(
    df,
    y="valorLiquido",
    points="all",
    title="Outliers Salariais"
)

fig.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

class AnalisadorServidores:
    def __init__(self, df):
        self.df = df

    def analise_remuneracao(self):
        print("ANALISE DE REMUNERACAO")
        print("Remuneracao Media por Sexo:")
        remuneracao_sexo = self.df.groupby('sexo').agg({
            'valorBruto': ['mean', 'median', 'std'],
            'valorLiquido': ['mean', 'median'],
            'nomeServidor': 'count'
        }).round(2)
        print(remuneracao_sexo)

        print("Remuneracao Media por Tipo de Cargo (Top 10):")
        remuneracao_cargo = self.df.groupby('tipoCargo').agg({
            'valorBruto': ['mean', 'count']
        }).round(2).sort_values(('valorBruto', 'mean'), ascending=False).head(10)
        print(remuneracao_cargo)

        print("Remuneracao Media por Regime Contratual:")
        remuneracao_regime = self.df.groupby('regimeContratual').agg({
            'valorBruto': ['mean', 'median'],
            'nomeServidor': 'count'
        }).round(2)
        print(remuneracao_regime)

    def analise_descontos(self):
        print("ANALISE DE DESCONTOS E ENCARGOS")
        print("Descontos Totais:")
        print(f"  Desconto medio: R$ {self.df['valorDesconto'].mean():.2f}")
        print(f"  Desconto mediano: R$ {self.df['valorDesconto'].median():.2f}")
        print(f"  Desconto maximo: R$ {self.df['valorDesconto'].max():.2f}")
        print(f"  Percentual de desconto medio: {self.df['descontosPercentual'].mean():.2f}%")

        print("Composicao dos Descontos (valores medios):")
        descontos_composicao = self.df[['valorPrevidenciario', 'valorIr', 'descontoObrigatorio']].mean()
        total_desconto_medio = self.df['valorDesconto'].mean()
        for coluna in descontos_composicao.index:
            percentual = (descontos_composicao[coluna] / total_desconto_medio * 100) if total_desconto_medio > 0 else 0
            print(f"  {coluna}: R$ {descontos_composicao[coluna]:.2f} ({percentual:.1f}%)")

        print("Descontos por Situacao do Servidor:")
        descontos_situacao = self.df.groupby('situacaoServidor').agg({
            'valorDesconto': ['mean', 'median'],
            'valorPrevidenciario': 'mean',
            'valorIr': 'mean'
        }).round(2)
        print(descontos_situacao)

    def analise_anos_servico(self):
        print("ANALISE DE ANOS DE SERVICO")
        print("Estatisticas de Anos de Servico:")
        print(f"  Minimo: {self.df['anosServico'].min():.1f} anos")
        print(f"  Maximo: {self.df['anosServico'].max():.1f} anos")
        print(f"  Media: {self.df['anosServico'].mean():.1f} anos")
        print(f"  Mediana: {self.df['anosServico'].median():.1f} anos")

        self.df['categoriaAnosServico'] = pd.cut(
            self.df['anosServico'],
            bins=[0, 1, 5, 10, 20, float('inf')],
            labels=['< 1 ano', '1-5 anos', '5-10 anos', '10-20 anos', '> 20 anos']
        )

        print("Distribuicao por Categorias de Tempo de Servico:")
        dist_tempo = self.df['categoriaAnosServico'].value_counts().sort_index()
        for categoria, quantidade in dist_tempo.items():
            percentual = (quantidade / len(self.df)) * 100
            print(f"  {categoria}: {quantidade} ({percentual:.1f}%)")

        print("Correlacao: Anos de Servico vs Remuneracao:")
        correlacao = self.df['anosServico'].corr(self.df['valorBruto'])
        print(f"  Correlacao de Pearson: {correlacao:.3f}")

    def analise_distribuicao_orgaos(self):
        print("ANALISE DE DISTRIBUICAO POR ORGAOS")
        print("Top 10 Orgaos por Quantidade de Servidores:")
        orgaos = self.df['orgaoLotacao'].value_counts().head(10)
        for orgao, quantidade in orgaos.items():
            percentual = (quantidade / len(self.df)) * 100
            print(f"  {orgao}: {quantidade} ({percentual:.1f}%)")

        print("Top 10 Orgaos por Folha de Pagamento:")
        folha_por_orgao = self.df.groupby('orgaoLotacao')['valorBruto'].sum().sort_values(ascending=False).head(10)
        for orgao, folha in folha_por_orgao.items():
            print(f"  {orgao}: R$ {folha:,.2f}")

    def analise_escolaridade(self):
        print("ANALISE POR ESCOLARIDADE")
        print("Distribuicao por Escolaridade Minima:")
        escolaridade = self.df['escolaridadeMinimaCargo'].value_counts()
        for nivel, quantidade in escolaridade.items():
            percentual = (quantidade / len(self.df)) * 100
            print(f"  {nivel}: {quantidade} ({percentual:.1f}%)")

        print("Remuneracao Media por Escolaridade:")
        remuneracao_escolaridade = self.df.groupby('escolaridadeMinimaCargo').agg({
            'valorBruto': ['mean', 'median', 'count']
        }).round(2).sort_values(('valorBruto', 'mean'), ascending=False)
        print(remuneracao_escolaridade)

    def analise_pessoas_deficientes(self):
        print("ANALISE - PESSOAS COM DEFICIENCIA")
        total = len(self.df)
        com_deficiencia = (self.df['deficienteFisico'] == 'SIM').sum()
        sem_deficiencia = (self.df['deficienteFisico'] == 'NAO').sum()

        print(f"Total de Servidores: {total}")
        print(f"  Com deficiencia: {com_deficiencia} ({com_deficiencia/total*100:.2f}%)")
        print(f"  Sem deficiencia: {sem_deficiencia} ({sem_deficiencia/total*100:.2f}%)")

        if com_deficiencia > 0:
            print("Cargos de Pessoas com Deficiencia:")
            cargos_def = self.df[self.df['deficienteFisico'] == 'SIM']['nomeCargo'].value_counts().head(5)
            for cargo, quantidade in cargos_def.items():
                print(f"  {cargo}: {quantidade}")

    def analise_administracao_indireta(self):
        print("ANALISE - ADMINISTRACAO INDIRETA")
        adm_indireta = self.df[self.df['administracao'] == 'ADMINISTRACAO INDIRETA']

        print(f"Total de Servidores em Admin. Indireta: {len(adm_indireta)}")
        print(f"   Percentual do total: {len(adm_indireta)/len(self.df)*100:.1f}%")

        print("Distribuicao por Entidade:")
        entidades = adm_indireta['cnpjOrgao'].value_counts()
        for entidade, quantidade in entidades.items():
            print(f"  {entidade}: {quantidade}")

        print("Folha de Pagamento:")
        print(f"  Total: R$ {adm_indireta['valorBruto'].sum():,.2f}")
        print(f"  Media por servidor: R$ {adm_indireta['valorBruto'].mean():,.2f}")

    def gerar_relatorio_completo(self):
        self.analise_remuneracao()
        self.analise_descontos()
        self.analise_anos_servico()
        self.analise_distribuicao_orgaos()
        self.analise_escolaridade()
        self.analise_pessoas_deficientes()
        self.analise_administracao_indireta()

if __name__ == "__main__":
    df = pd.read_csv('dataset_preparado.csv', sep=';', encoding='utf-8')
    df['dataAdmissao'] = pd.to_datetime(df['dataAdmissao'])
    df['anosServico'] = (datetime.now() - df['dataAdmissao']).dt.days / 365.25
    analisador = AnalisadorServidores(df)
    analisador.gerar_relatorio_completo()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

class AnalisadorServidores:
    def __init__(self, df):
        self.df = df

    def analise_remuneracao(self):
        print("Remuneracao Media por Sexo:")
        remuneracao_sexo = self.df.groupby('sexo').agg({
            'valorBruto': ['mean', 'median', 'std'],
            'valorLiquido': ['mean', 'median'],
            'nomeServidor': 'count'
        }).round(2)
        print(remuneracao_sexo)

        print("Remuneracao Media por Tipo de Cargo (Top 10):")
        remuneracao_cargo = self.df.groupby('tipoCargo').agg({
            'valorBruto': ['mean', 'count']
        }).round(2).sort_values(('valorBruto', 'mean'), ascending=False).head(10)
        print(remuneracao_cargo)

        print("Remuneracao Media por Regime Contratual:")
        remuneracao_regime = self.df.groupby('regimeContratual').agg({
            'valorBruto': ['mean', 'median'],
            'nomeServidor': 'count'
        }).round(2)
        print(remuneracao_regime)

    def analise_descontos(self):
        print("Descontos Totais:")
        print(f"  Desconto medio: R$ {self.df['valorDesconto'].mean():.2f}")
        print(f"  Desconto mediano: R$ {self.df['valorDesconto'].median():.2f}")
        print(f"  Desconto maximo: R$ {self.df['valorDesconto'].max():.2f}")
        print(f"  Percentual de desconto medio: {self.df['descontosPercentual'].mean():.2f}%")

        print("Composicao dos Descontos (valores medios):")
        descontos_composicao = self.df[['valorPrevidenciario', 'valorIr', 'descontoObrigatorio']].mean()
        total_desconto_medio = self.df['valorDesconto'].mean()
        for coluna in descontos_composicao.index:
            percentual = (descontos_composicao[coluna] / total_desconto_medio * 100) if total_desconto_medio > 0 else 0
            print(f"  {coluna}: R$ {descontos_composicao[coluna]:.2f} ({percentual:.1f}%)")

        print("Descontos por Situacao do Servidor:")
        descontos_situacao = self.df.groupby('situacaoServidor').agg({
            'valorDesconto': ['mean', 'median'],
            'valorPrevidenciario': 'mean',
            'valorIr': 'mean'
        }).round(2)
        print(descontos_situacao)

    def analise_anos_servico(self):
        print("Estatisticas de Anos de Servico:")
        print(f"  Minimo: {self.df['anosServico'].min():.1f} anos")
        print(f"  Maximo: {self.df['anosServico'].max():.1f} anos")
        print(f"  Media: {self.df['anosServico'].mean():.1f} anos")
        print(f"  Mediana: {self.df['anosServico'].median():.1f} anos")

        self.df['categoriaAnosServico'] = pd.cut(
            self.df['anosServico'],
            bins=[0, 1, 5, 10, 20, float('inf')],
            labels=['< 1 ano', '1-5 anos', '5-10 anos', '10-20 anos', '> 20 anos']
        )

        print("Distribuicao por Categorias de Tempo de Servico:")
        dist_tempo = self.df['categoriaAnosServico'].value_counts().sort_index()
        for categoria, quantidade in dist_tempo.items():
            percentual = (quantidade / len(self.df)) * 100
            print(f"  {categoria}: {quantidade} ({percentual:.1f}%)")

        print("Correlacao: Anos de Servico vs Remuneracao:")
        correlacao = self.df['anosServico'].corr(self.df['valorBruto'])
        print(f"  Correlacao de Pearson: {correlacao:.3f}")

    def analise_distribuicao_orgaos(self):
        print("Top 10 Orgaos por Quantidade de Servidores:")
        orgaos = self.df['orgaoLotacao'].value_counts().head(10)
        for orgao, quantidade in orgaos.items():
            percentual = (quantidade / len(self.df)) * 100
            print(f"  {orgao}: {quantidade} ({percentual:.1f}%)")

        print("Top 10 Orgaos por Folha de Pagamento:")
        folha_por_orgao = self.df.groupby('orgaoLotacao')['valorBruto'].sum().sort_values(ascending=False).head(10)
        for orgao, folha in folha_por_orgao.items():
            print(f"  {orgao}: R$ {folha:,.2f}")

    def analise_escolaridade(self):
        print("Distribuicao por Escolaridade Minima:")
        escolaridade = self.df['escolaridadeMinimaCargo'].value_counts()
        for nivel, quantidade in escolaridade.items():
            percentual = (quantidade / len(self.df)) * 100
            print(f"  {nivel}: {quantidade} ({percentual:.1f}%)")

        print("Remuneracao Media por Escolaridade:")
        remuneracao_escolaridade = self.df.groupby('escolaridadeMinimaCargo').agg({
            'valorBruto': ['mean', 'median', 'count']
        }).round(2).sort_values(('valorBruto', 'mean'), ascending=False)
        print(remuneracao_escolaridade)

    def analise_pessoas_deficientes(self):
        total = len(self.df)
        com_deficiencia = (self.df['deficienteFisico'] == 'SIM').sum()
        sem_deficiencia = (self.df['deficienteFisico'] == 'NAO').sum()
        print(f"Total de Servidores: {total}")
        print(f"  Com deficiencia: {com_deficiencia} ({com_deficiencia/total*100:.2f}%)")
        print(f"  Sem deficiencia: {sem_deficiencia} ({sem_deficiencia/total*100:.2f}%)")
        if com_deficiencia > 0:
            print("Cargos de Pessoas com Deficiencia:")
            cargos_def = self.df[self.df['deficienteFisico'] == 'SIM']['nomeCargo'].value_counts().head(5)
            for cargo, quantidade in cargos_def.items():
                print(f"  {cargo}: {quantidade}")

    def analise_administracao_indireta(self):
        adm_indireta = self.df[self.df['administracao'] == 'ADMINISTRACAO INDIRETA']
        print(f"Total de Servidores em Admin. Indireta: {len(adm_indireta)}")
        print(f"   Percentual do total: {len(adm_indireta)/len(self.df)*100:.1f}%")
        print("Distribuicao por Entidade:")
        entidades = adm_indireta['cnpjOrgao'].value_counts()
        for entidade, quantidade in entidades.items():
            print(f"  {entidade}: {quantidade}")
        print("Folha de Pagamento:")
        print(f"  Total: R$ {adm_indireta['valorBruto'].sum():,.2f}")
        print(f"  Media por servidor: R$ {adm_indireta['valorBruto'].mean():,.2f}")

    def gerar_relatorio_completo(self):
        self.analise_remuneracao()
        self.analise_descontos()
        self.analise_anos_servico()
        self.analise_distribuicao_orgaos()
        self.analise_escolaridade()
        self.analise_pessoas_deficientes()
        self.analise_administracao_indireta()

if __name__ == "__main__":
    df = pd.read_csv('dataset_preparado.csv', sep=';', encoding='utf-8')
    df['dataAdmissao'] = pd.to_datetime(df['dataAdmissao'])
    df['anosServico'] = (datetime.now() - df['dataAdmissao']).dt.days / 365.25
    analisador = AnalisadorServidores(df)
    analisador.gerar_relatorio_completo()

In [ ]:
%pip install --upgrade nbformat

In [ ]:
import requests
import csv
import os

class SEADParaibaCSV:
    def __init__(self):
        self.base_url = "https://api.dadosabertos.codata.pb.gov.br/api/v1/remuneracao"
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/json",
            "Content-Type": "application/json"
        })

    def buscar_pagina(self, ano, mes, pagina):
        url = f"{self.base_url}/servidor"
        params = {
            "ano": str(ano),
            "mes": str(mes).zfill(2),
            "page": pagina
        }
        
        response = self.session.get(url, params=params)
        
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Erro ao buscar página {pagina}. Status: {response.status_code}")
            return None

    def localizar_lista(self, dados_json):
        if isinstance(dados_json, list):
            return dados_json
            
        if isinstance(dados_json, dict):
            for chave, valor in dados_json.items():
                if isinstance(valor, list):
                    return valor
        return []

    def exportar_para_csv(self, ano, mes):
        nome_arquivo = f"servidores_pb_{ano}_{str(mes).zfill(2)}.csv"
        
        print(f"Iniciando extração. Os dados serão salvos em: {nome_arquivo}\n")

        arquivo_csv = None
        escritor = None
        cabecalhos = []
        pagina = 1

        while True:
            print(f"Baixando página {pagina}...")
            dados = self.buscar_pagina(ano, mes, pagina)
            
            if not dados:
                print("Falha na requisição. Encerrando busca.")
                break
                
            lista_servidores = self.localizar_lista(dados)
            
            if not lista_servidores:
                print(f"A lista de servidores na página {pagina} veio vazia. Fim da extração.")
                break
                
            for servidor in lista_servidores:
                if not cabecalhos:
                    cabecalhos = list(servidor.keys())
                    
                    arquivo_csv = open(nome_arquivo, mode="w", newline="", encoding="utf-8-sig")
                    escritor = csv.DictWriter(arquivo_csv, fieldnames=cabecalhos, delimiter=";")
                    escritor.writeheader()

                escritor.writerow(servidor)
            
            pagina += 1

        if arquivo_csv:
            arquivo_csv.close()

        total_paginas = pagina - 1
        print(f"\nExtração concluída com sucesso! Total de páginas processadas: {total_paginas}")
        print(f"Arquivo salvo no caminho: {os.path.abspath(nome_arquivo)}")

if __name__ == "__main__":
    extrator = SEADParaibaCSV()
    extrator.exportar_para_csv(ano=2026, mes=4)

In [ ]:
import requests
import logging
from typing import Dict, List, Any, Optional
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class DadosAbertosPBAPI:
    """
    Cliente para a API Dados Abertos PB.
    
    Fornece acesso aos dados públicos da Paraíba em categorias como:
    - Servidores e remuneração
    - Despesas e notas de empenho
    - Compras, contratações e contratos
    """

    def __init__(self, base_url: str = "https://api.dadosabertos.codata.pb.gov.br/api/v1", 
                 timeout: int = 30, max_retries: int = 3):
        """
        Inicializa o cliente da API.
        
        Args:
            base_url: URL base da API
            timeout: Tempo limite para requisições (segundos)
            max_retries: Número máximo de tentativas para requisições
        """
        self.base_url = base_url
        self.timeout = timeout
        self.session = self._criar_sessao_com_retry(max_retries)

    def _criar_sessao_com_retry(self, max_retries: int) -> requests.Session:
        """
        Cria uma sessão com estratégia de retry automático.
        
        Args:
            max_retries: Número máximo de tentativas
            
        Returns:
            Session configurada com retry strategy
        """
        sessao = requests.Session()
        sessao.headers.update({"Accept": "application/json"})
        
        # Configurar retry strategy para conexões falhas
        retry_strategy = Retry(
            total=max_retries,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"]
        )
        
        adapter = HTTPAdapter(max_retries=retry_strategy)
        sessao.mount("http://", adapter)
        sessao.mount("https://", adapter)
        
        return sessao

    def _get(self, endpoint: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        """
        Realiza uma requisição GET à API.
        
        Args:
            endpoint: Caminho do endpoint (sem base_url)
            params: Parâmetros da query
            
        Returns:
            Response JSON da API
            
        Raises:
            requests.RequestException: Em caso de erro na requisição
        """
        url = f"{self.base_url}/{endpoint}"
        
        try:
            logger.debug(f"GET {url} com params: {params}")
            response = self.session.get(url, params=params, timeout=self.timeout)
            response.raise_for_status()
            return response.json()
        
        except requests.exceptions.Timeout:
            logger.error(f"Timeout ao acessar {url}")
            raise
        except requests.exceptions.ConnectionError as e:
            logger.error(f"Erro de conexão ao acessar {url}: {e}")
            raise
        except requests.exceptions.HTTPError as e:
            logger.error(f"Erro HTTP {response.status_code} ao acessar {url}: {response.text}")
            raise
        except ValueError as e:
            logger.error(f"Erro ao decodificar JSON de {url}: {e}")
            raise

    def _listar_paginado(self, endpoint: str, ano: int = 2026, mes: Optional[int] = None,
                         pagina: int = 1) -> Dict[str, Any]:
        """
        Método genérico para listar dados com paginação.
        
        Args:
            endpoint: Caminho do endpoint
            ano: Ano do exercício
            mes: Mês (opcional)
            pagina: Número da página
            
        Returns:
            Response com dados e informações de paginação
        """
        params = {"page": pagina}
        
        if ano:
            params["ano"] = ano
        if mes:
            params["mes"] = str(mes).zfill(2)
        
        return self._get(endpoint, params)

    def _obter_todos_registros(self, endpoint: str, ano: int, mes: Optional[int] = None) -> List[Dict[str, Any]]:
        """
        Obtém todos os registros de um endpoint paginado.
        
        Args:
            endpoint: Caminho do endpoint
            ano: Ano do exercício
            mes: Mês (opcional)
            
        Returns:
            Lista com todos os registros
        """
        pagina = 1
        todos_registros = []

        while True:
            try:
                resposta = self._listar_paginado(endpoint, ano, mes, pagina)
                
                registros = resposta.get("dados", [])
                paginacao = resposta.get("paginacao", {})
                
                todos_registros.extend(registros)
                
                total_paginas = paginacao.get("total_paginas", 1)
                logger.info(
                    f"Página {pagina}/{total_paginas} - "
                    f"Total acumulado: {len(todos_registros)} registros"
                )
                
                if pagina >= total_paginas:
                    break
                
                pagina += 1
            
            except requests.RequestException as e:
                logger.error(f"Erro ao buscar página {pagina}: {e}")
                raise

        return todos_registros

    # ==================== ENDPOINTS DE SERVIDORES ====================
    
    def listar_servidores(self, ano: int = 2026, mes: int = 1, pagina: int = 1) -> Dict[str, Any]:
        """
        Lista servidores públicos com dados de remuneração.
        
        Args:
            ano: Ano do exercício
            mes: Mês
            pagina: Número da página
            
        Returns:
            Dados de servidores paginados
        """
        return self._listar_paginado("remuneracao/servidor", ano, mes, pagina)

    def obter_todos_servidores(self, ano: int = 2026, mes: int = 1) -> List[Dict[str, Any]]:
        """Obtém todos os servidores (todas as páginas)."""
        return self._obter_todos_registros("remuneracao/servidor", ano, mes)

    # ==================== ENDPOINTS DE DESPESAS ====================
    
    def listar_notas_empenho(self, ano: int = 2026, mes: int = 5, pagina: int = 1) -> Dict[str, Any]:
        """
        Lista notas de empenho (comprometimento de despesa).
        
        Args:
            ano: Ano do exercício
            mes: Mês
            pagina: Número da página
            
        Returns:
            Dados de notas de empenho paginados
        """
        return self._listar_paginado("despesas/notas_empenho", ano, mes, pagina)

    def obter_todas_notas_empenho(self, ano: int = 2026, mes: int = 5) -> List[Dict[str, Any]]:
        """Obtém todas as notas de empenho (todas as páginas)."""
        return self._obter_todos_registros("despesas/notas_empenho", ano, mes)

    def listar_liquidacoes(self, ano: int = 2026, mes: int = 5, pagina: int = 1) -> Dict[str, Any]:
        """
        Lista liquidações de despesas.
        
        Args:
            ano: Ano do exercício
            mes: Mês
            pagina: Número da página
            
        Returns:
            Dados de liquidações paginados
        """
        return self._listar_paginado("despesas/liquidacoes", ano, mes, pagina)

    def obter_todas_liquidacoes(self, ano: int = 2026, mes: int = 5) -> List[Dict[str, Any]]:
        """Obtém todas as liquidações (todas as páginas)."""
        return self._obter_todos_registros("despesas/liquidacoes", ano, mes)

    # ==================== ENDPOINTS DE COMPRAS ====================
    
    def listar_contratacoes(self, ano: int = 2026, mes: int = 5, pagina: int = 1) -> Dict[str, Any]:
        """
        Lista contratações/licitações.
        
        Args:
            ano: Ano do exercício
            mes: Mês
            pagina: Número da página
            
        Returns:
            Dados de contratações paginados
        """
        return self._listar_paginado("compras/contratacoes", ano, mes, pagina)

    def obter_todas_contratacoes(self, ano: int = 2026, mes: int = 5) -> List[Dict[str, Any]]:
        """Obtém todas as contratações (todas as páginas)."""
        return self._obter_todos_registros("compras/contratacoes", ano, mes)

    def listar_contratos(self, ano: int = 2026, mes: int = 5, pagina: int = 1) -> Dict[str, Any]:
        """
        Lista contratos celebrados.
        
        Args:
            ano: Ano do exercício
            mes: Mês
            pagina: Número da página
            
        Returns:
            Dados de contratos paginados
        """
        return self._listar_paginado("compras/contratos", ano, mes, pagina)

    def obter_todos_contratos(self, ano: int = 2026, mes: int = 5) -> List[Dict[str, Any]]:
        """Obtém todos os contratos (todas as páginas)."""
        return self._obter_todos_registros("compras/contratos", ano, mes)

    def listar_itens_contratacoes(self, ano: int = 2026, mes: int = 5, pagina: int = 1) -> Dict[str, Any]:
        """
        Lista itens de contratações.
        
        Args:
            ano: Ano do exercício
            mes: Mês
            pagina: Número da página
            
        Returns:
            Dados de itens paginados
        """
        return self._listar_paginado("compras/itens_contratacoes", ano, mes, pagina)

    def obter_todos_itens_contratacoes(self, ano: int = 2026, mes: int = 5) -> List[Dict[str, Any]]:
        """Obtém todos os itens de contratações (todas as páginas)."""
        return self._obter_todos_registros("compras/itens_contratacoes", ano, mes)

    def listar_ata_registro_preco(self, ano: int = 2026, mes: int = 5, pagina: int = 1) -> Dict[str, Any]:
        """
        Lista atas de registro de preço.
        
        Args:
            ano: Ano do exercício
            mes: Mês
            pagina: Número da página
            
        Returns:
            Dados de atas paginados
        """
        return self._listar_paginado("compras/ata_registro_preco", ano, mes, pagina)

    def obter_todas_atas_registro_preco(self, ano: int = 2026, mes: int = 5) -> List[Dict[str, Any]]:
        """Obtém todas as atas de registro de preço (todas as páginas)."""
        return self._obter_todos_registros("compras/ata_registro_preco", ano, mes)


# ==================== EXEMPLO DE USO ====================

# if __name__ == "__main__":
#     # Exemplo básico
#     api = DadosAbertosPBAPI()
    
#     try:
#         # Obter primeira página de notas de empenho
#         notas = api.listar_notas_empenho(ano=2026, mes=5, pagina=1)
#         print(f"Primeira página: {len(notas.get('dados', []))} registros encontrados")
        
#         # Obter todas as notas de empenho (pode levar tempo)
#         # todas_notas = api.obter_todas_notas_empenho(ano=2026, mes=5)
#         # print(f"Total de notas: {len(todas_notas)}")
        
#     except requests.RequestException as e:
#         logger.error(f"Erro ao conectar à API: {e}")

In [ ]:
"""
Extrator de Dados - API Dados Abertos PB

Módulo para extrair dados da API Dados Abertos Paraíba e salvar em múltiplos formatos.
Suporta extração por período, filtragem e processamento de dados.
"""

import logging
import json
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from datetime import datetime
import pandas as pd
import numpy as np
from requests.exceptions import RequestException

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class ExtractorNotasEmpenho:
    """
    Extrator de Notas de Empenho da API Dados Abertos PB.
    
    Extrai dados de notas de empenho por período (ano/mês) e salva em múltiplos formatos.
    """

    def __init__(self, api: Optional[DadosAbertosPBAPI] = None, 
                 diretorio_saida: str = "."):
        """
        Inicializa o extrator.
        
        Args:
            api: Instância de DadosAbertosPBAPI (cria nova se não fornecida)
            diretorio_saida: Diretório para salvar arquivos
        """
        self.api = api or DadosAbertosPBAPI()
        self.diretorio_saida = Path(diretorio_saida)
        self.diretorio_saida.mkdir(parents=True, exist_ok=True)
        
        # Estatísticas
        self.stats = {
            "total_registros": 0,
            "anos_processados": [],
            "meses_processados": [],
            "erros": []
        }

    def _validar_periodo(self, ano: int, mes: Optional[int] = None) -> bool:
        """
        Valida se o período é válido.
        
        Args:
            ano: Ano (deve ser >= 2000)
            mes: Mês (1-12, opcional)
            
        Returns:
            True se válido, False caso contrário
        """
        if ano < 2000:
            logger.warning(f"Ano inválido: {ano}")
            return False
        
        if mes is not None and (mes < 1 or mes > 12):
            logger.warning(f"Mês inválido: {mes}")
            return False
        
        return True

    def extrair_mes(self, ano: int, mes: int, 
                    verbose: bool = True) -> List[Dict[str, Any]]:
        """
        Extrai dados de um mês específico.
        
        Args:
            ano: Ano
            mes: Mês (1-12)
            verbose: Exibir detalhes
            
        Returns:
            Lista com dados extraídos
        """
        if not self._validar_periodo(ano, mes):
            return []
        
        try:
            if verbose:
                logger.info(f"🔍 Extraindo {ano}-{mes:02d}...")
            
            dados = self.api.obter_todas_notas_empenho(ano=ano, mes=mes)
            
            if verbose:
                logger.info(f"✅ Extraído {ano}-{mes:02d}: {len(dados)} registros")
            
            return dados
        
        except RequestException as e:
            msg = f"Erro de requisição em {ano}-{mes:02d}: {e}"
            logger.error(msg)
            self.stats["erros"].append((ano, mes, str(e)))
            return []
        
        except Exception as e:
            msg = f"Erro inesperado em {ano}-{mes:02d}: {e}"
            logger.error(msg)
            self.stats["erros"].append((ano, mes, str(e)))
            return []

    def extrair_ano(self, ano: int, verbose: bool = True) -> Tuple[List[Dict], int]:
        """
        Extrai dados de um ano completo (12 meses).
        
        Args:
            ano: Ano
            verbose: Exibir detalhes
            
        Returns:
            Tupla (dados, total_registros)
        """
        if not self._validar_periodo(ano):
            return [], 0
        
        logger.info(f"\n{'='*60}")
        logger.info(f"🗓️  Extraindo dados de {ano}")
        logger.info(f"{'='*60}")
        
        dados_ano = []
        
        for mes in range(1, 13):
            dados_mes = self.extrair_mes(ano, mes, verbose=verbose)
            dados_ano.extend(dados_mes)
        
        total = len(dados_ano)
        self.stats["total_registros"] += total
        self.stats["anos_processados"].append(ano)
        
        logger.info(f"✨ Total para {ano}: {total} registros")
        
        return dados_ano, total

    def extrair_periodo(self, ano_inicio: int, ano_fim: int,
                       verbose: bool = True) -> Tuple[List[Dict], int]:
        """
        Extrai dados de um período (range de anos).
        
        Args:
            ano_inicio: Ano inicial (inclusive)
            ano_fim: Ano final (inclusive)
            verbose: Exibir detalhes
            
        Returns:
            Tupla (dados, total_registros)
        """
        if ano_inicio > ano_fim:
            logger.error("Ano inicial deve ser menor ou igual ao ano final")
            return [], 0
        
        todos_dados = []
        
        for ano in range(ano_inicio, ano_fim + 1):
            dados, _ = self.extrair_ano(ano, verbose=verbose)
            todos_dados.extend(dados)
        
        return todos_dados, len(todos_dados)

    def _salvar_csv(self, df: pd.DataFrame, nome_arquivo: str) -> Path:
        """Salva DataFrame em CSV."""
        caminho = self.diretorio_saida / nome_arquivo
        df.to_csv(caminho, index=False, encoding="utf-8-sig")
        logger.info(f"💾 Salvo CSV: {caminho}")
        return caminho

    def _salvar_excel(self, df: pd.DataFrame, nome_arquivo: str) -> Path:
        """Salva DataFrame em Excel."""
        try:
            caminho = self.diretorio_saida / nome_arquivo
            
            with pd.ExcelWriter(caminho, engine='openpyxl') as writer:
                df.to_excel(writer, index=False, sheet_name='Dados')
                
                # Auto-ajustar largura das colunas
                worksheet = writer.sheets['Dados']
                for idx, col in enumerate(df.columns):
                    max_length = max(
                        df[col].astype(str).str.len().max(),
                        len(str(col))
                    )
                    worksheet.column_dimensions[
                        chr(65 + idx)
                    ].width = min(max_length + 2, 50)
            
            logger.info(f"💾 Salvo Excel: {caminho}")
            return caminho
        
        except ImportError:
            logger.warning("openpyxl não instalado, salvando como CSV")
            return self._salvar_csv(df, nome_arquivo.replace('.xlsx', '.csv'))

    def _salvar_json(self, dados: List[Dict], nome_arquivo: str) -> Path:
        """Salva dados em JSON."""
        caminho = self.diretorio_saida / nome_arquivo
        
        with open(caminho, 'w', encoding='utf-8') as f:
            json.dump(dados, f, indent=2, ensure_ascii=False, default=str)
        
        logger.info(f"💾 Salvo JSON: {caminho}")
        return caminho

    def _gerar_relatorio(self, df: pd.DataFrame, caminho_base: str) -> Path:
        """Gera relatório estatístico dos dados."""
        caminho = self.diretorio_saida / f"{caminho_base}_relatorio.txt"
        
        with open(caminho, 'w', encoding='utf-8') as f:
            f.write("="*60 + "\n")
            f.write("RELATÓRIO DE EXTRAÇÃO - NOTAS DE EMPENHO\n")
            f.write("="*60 + "\n\n")
            
            f.write(f"Data/Hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total de registros: {len(df)}\n")
            f.write(f"Tamanho em memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB\n\n")
            
            f.write("COLUNAS:\n")
            f.write("-"*60 + "\n")
            for col in df.columns:
                dtype = df[col].dtype
                nulos = df[col].isna().sum()
                f.write(f"  {col}: {dtype} ({nulos} nulos)\n")
            
            f.write("\nESTATÍSTICAS:\n")
            f.write("-"*60 + "\n")
            f.write(df.describe(include='all').to_string())
            f.write("\n\nERROS ENCONTRADOS:\n")
            f.write("-"*60 + "\n")
            if self.stats["erros"]:
                for ano, mes, erro in self.stats["erros"]:
                    f.write(f"  {ano}-{mes:02d}: {erro}\n")
            else:
                f.write("  Nenhum erro!\n")
        
        logger.info(f"📊 Relatório: {caminho}")
        return caminho

    def salvar_dados(self, dados: List[Dict], ano: int,
                     formatos: List[str] = None,
                     incluir_relatorio: bool = True) -> Dict[str, Path]:
        """
        Salva dados em múltiplos formatos.
        
        Args:
            dados: Lista de dados
            ano: Ano (para nomeação)
            formatos: Lista com formatos desejados ('csv', 'excel', 'json')
            incluir_relatorio: Gerar relatório estatístico
            
        Returns:
            Dicionário com caminhos dos arquivos salvos
        """
        if not dados:
            logger.warning(f"Nenhum dado para salvar do ano {ano}")
            return {}
        
        formatos = formatos or ['csv']
        df = pd.DataFrame(dados)
        
        arquivos_salvos = {}
        caminho_base = f"notas_empenho_{ano}"
        
        for formato in formatos:
            try:
                if formato.lower() == 'csv':
                    arquivo = f"{caminho_base}.csv"
                    caminho = self._salvar_csv(df, arquivo)
                    arquivos_salvos['csv'] = caminho
                
                elif formato.lower() == 'excel':
                    arquivo = f"{caminho_base}.xlsx"
                    caminho = self._salvar_excel(df, arquivo)
                    arquivos_salvos['excel'] = caminho
                
                elif formato.lower() == 'json':
                    arquivo = f"{caminho_base}.json"
                    caminho = self._salvar_json(dados, arquivo)
                    arquivos_salvos['json'] = caminho
                
                else:
                    logger.warning(f"Formato desconhecido: {formato}")
            
            except Exception as e:
                logger.error(f"Erro ao salvar {formato}: {e}")
        
        if incluir_relatorio and not df.empty:
            self._gerar_relatorio(df, caminho_base)
        
        return arquivos_salvos

    def processar_periodo_com_salvamento(self, 
                                        ano_inicio: int, 
                                        ano_fim: int,
                                        formatos: List[str] = None,
                                        incluir_relatorio: bool = True,
                                        salvar_por_ano: bool = True) -> Dict[int, Dict]:
        """
        Processa período completo e salva dados.
        
        Args:
            ano_inicio: Ano inicial
            ano_fim: Ano final
            formatos: Formatos de saída
            incluir_relatorio: Gerar relatórios
            salvar_por_ano: Salvar arquivo separado por ano
            
        Returns:
            Dicionário com informações dos arquivos salvos
        """
        formatos = formatos or ['csv']
        resultados = {}
        todos_dados = []
        
        for ano in range(ano_inicio, ano_fim + 1):
            dados_ano, total = self.extrair_ano(ano, verbose=True)
            todos_dados.extend(dados_ano)
            
            if salvar_por_ano and dados_ano:
                logger.info(f"\n💾 Salvando dados de {ano}...")
                arquivos = self.salvar_dados(
                    dados_ano,
                    ano,
                    formatos=formatos,
                    incluir_relatorio=incluir_relatorio
                )
                resultados[ano] = {
                    "total_registros": total,
                    "arquivos": arquivos
                }
        
        # Salvar consolidado
        if todos_dados and ano_fim - ano_inicio > 0:
            logger.info(f"\n💾 Salvando dados consolidados ({ano_inicio}-{ano_fim})...")
            periodo = f"{ano_inicio}_{ano_fim}"
            arquivos_consolidados = self.salvar_dados(
                todos_dados,
                periodo,
                formatos=formatos,
                incluir_relatorio=incluir_relatorio
            )
            resultados["consolidado"] = {
                "total_registros": len(todos_dados),
                "arquivos": arquivos_consolidados
            }
        
        return resultados

    def exibir_resumo(self) -> None:
        """Exibe resumo das estatísticas de extração."""
        print("\n" + "="*60)
        print(" RESUMO DE EXTRAÇÃO")
        print("="*60)
        print(f"Total de registros: {self.stats['total_registros']:,}")
        print(f"Anos processados: {len(self.stats['anos_processados'])}")
        
        if self.stats['anos_processados']:
            print(f"  • {', '.join(map(str, sorted(self.stats['anos_processados'])))}")
        
        print(f"Erros encontrados: {len(self.stats['erros'])}")
        if self.stats['erros']:
            for ano, mes, erro in self.stats['erros'][:5]:  # Exibir apenas 5
                print(f"  • {ano}-{mes:02d}: {erro[:50]}...")
        
        print("="*60 + "\n")


# ==================== EXEMPLOS DE USO ====================
def testing_extrator():
    extrator = ExtractorNotasEmpenho(diretorio_saida="./dados")
    result = extrator.processar_periodo_com_salvamento(2020, 2026, formatos=['csv'], incluir_relatorio=True, salvar_por_ano=True)
    print("\nExtração completa. Resumo:")
    extrator.exibir_resumo()
    print(result)

In [ ]:
testing_extrator()

In [ ]:
if __name__ == "__main__":
    api = DadosAbertosPBAPI()

    dados = api.listar_servidores(ano=2026, mes=5, pagina=1)

    print(type(dados))

    if isinstance(dados, dict):
        print(dados.keys())

        for chave, valor in dados.items():
            if isinstance(valor, list):
                print(f"Lista encontrada: {chave}")
                print(f"Quantidade: {len(valor)}")

                if valor:
                    print(valor[0])
                break

In [ ]:
import pandas as pd

todos = []
api = DadosAbertosPBAPI()

for ano in range(2026, 2027):
    # Lista para acumular os dados apenas do ano corrente
    dados_ano = []
    
    for mes in range(1, 13):
        try:
            dados = api.obter_todas_notas_empenho(
                ano=ano,
                mes=mes
            )

            # Adiciona na lista geral (se você for usar 'todos' depois)
            todos.extend(dados)
            
            # Adiciona na lista específica deste ano
            dados_ano.extend(dados)

            print(f"Extraído {ano}-{mes:02d}: {len(dados)} registros")

        except Exception as e:
            print(f"Erro em {ano}-{mes:02d}: {e}")
            
    # --- SALVAMENTO POR ANO ---
    # Ao terminar os 12 meses, converte os dados acumulados do ano e salva
    if dados_ano: # Verifica se a lista não está vazia
        df = pd.DataFrame(dados_ano)

        df.to_csv(
            f"notas_empenho_{ano}.csv",
            index=False,
            encoding="utf-8-sig"
        )
        print(f">>> Arquivo salvo: notas_empenho_{ano}.csv com {len(dados_ano)} registros totais! <<<\n")